In [2]:
import pandas as pd

df = pd.read_csv("data/tfl_journeys_2024.csv")

print(df.head())

         Date    Journey_ID Transport_Mode Origin_Station Destination_Station  \
0  21/09/2024  TFL-J-422610           Tube      Stratford    Liverpool Street   
1  14/02/2024  TFL-J-228818            Bus     Paddington           Stratford   
2  25/08/2024  TFL-J-723520           Tube         Euston          Paddington   
3  15/02/2024  TFL-J-695985     Overground     Paddington            Victoria   
4  03/11/2024  TFL-J-175457           Tube         Euston           Stratford   

        Tap_In_Time      Tap_Out_Time  Journey_Cost  Delay_Minutes  \
0  21/09/2024 12:03  21/09/2024 12:24          3.97              0   
1  14/02/2024 09:11  14/02/2024 10:02          1.75             20   
2  25/08/2024 08:03  25/08/2024 08:05          9.40             14   
3  15/02/2024 21:24  15/02/2024 22:05          2.81              2   
4  03/11/2024 07:12  03/11/2024 07:36          2.98              3   

  Origin_Borough Destination_Borough  
0         Newham      City of London  
1    Westminst

In [4]:
#Data Quality Checks
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

Date                   0
Journey_ID             0
Transport_Mode         0
Origin_Station         0
Destination_Station    0
Tap_In_Time            0
Tap_Out_Time           0
Journey_Cost           0
Delay_Minutes          0
Origin_Borough         0
Destination_Borough    0
dtype: int64

Duplicates:
0


In [5]:
#Convert Date Columns
df['Date'] = pd.to_datetime(
    df['Date'],
    dayfirst=True,
    errors='coerce'
)

df['Tap_In_Time'] = pd.to_datetime(
    df['Tap_In_Time'],
    dayfirst=True,
    errors='coerce'
)

df['Tap_Out_Time'] = pd.to_datetime(
    df['Tap_Out_Time'],
    dayfirst=True,
    errors='coerce'
)

In [6]:
#Dataset Information
print("\nDataset Info:")
print(df.info())

print("\nPreview:")
print(df.head())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 2000 non-null   datetime64[ns]
 1   Journey_ID           2000 non-null   object        
 2   Transport_Mode       2000 non-null   object        
 3   Origin_Station       2000 non-null   object        
 4   Destination_Station  2000 non-null   object        
 5   Tap_In_Time          2000 non-null   datetime64[ns]
 6   Tap_Out_Time         2000 non-null   datetime64[ns]
 7   Journey_Cost         2000 non-null   float64       
 8   Delay_Minutes        2000 non-null   int64         
 9   Origin_Borough       2000 non-null   object        
 10  Destination_Borough  2000 non-null   object        
dtypes: datetime64[ns](3), float64(1), int64(1), object(6)
memory usage: 172.0+ KB
None

Preview:
        Date    Journey_ID Transport_Mo

In [7]:
#Create Journey Duration
df['Journey_Duration'] = (
    df['Tap_Out_Time']
    -
    df['Tap_In_Time']
).dt.total_seconds() / 60

In [8]:
print(
    df[
        ['Journey_ID','Journey_Duration']
    ].head()
)

     Journey_ID  Journey_Duration
0  TFL-J-422610              21.0
1  TFL-J-228818              51.0
2  TFL-J-723520               2.0
3  TFL-J-695985              41.0
4  TFL-J-175457              24.0


In [9]:
#Find Anomalies
anomalies = df[
    df['Journey_Duration'] < 1
]

print("Anomalous Records:")
print(anomalies)

print(
    "\nTotal Anomalies Found:",
    anomalies.shape[0]
)

Anomalous Records:
           Date    Journey_ID Transport_Mode Origin_Station  \
9    2024-02-27  TFL-J-294616     Overground     Paddington   
17   2024-07-21  TFL-J-953660            Bus      Stratford   
33   2024-07-26  TFL-J-275876           Tube      Stratford   
36   2024-10-13  TFL-J-903833           Tube     Paddington   
91   2024-07-19  TFL-J-901349            Bus       Victoria   
...         ...           ...            ...            ...   
1940 2024-10-03  TFL-J-165452     Overground           Bank   
1947 2024-12-08  TFL-J-970062           Tube      Stratford   
1975 2024-03-07  TFL-J-772810           Tube       Victoria   
1982 2024-05-03  TFL-J-318713           Tube     Paddington   
1996 2024-08-05  TFL-J-915004           Tube           Bank   

     Destination_Station         Tap_In_Time        Tap_Out_Time  \
9               Waterloo 2024-02-27 18:32:00 2024-02-27 17:38:00   
17              Waterloo 2024-07-21 18:24:00 2024-07-21 18:24:00   
33                  

In [10]:
#Save Quarantined Data
anomalies.to_csv(
    "quarantined_data.csv",
    index=False
)

In [20]:
#Create Clean Dataset
df_clean = df[
    ~(df['Journey_Duration'] < 1)
].copy()

print(df_clean.shape)

(1925, 12)


In [21]:
df_clean.to_csv(
    "cleaned_tfl_data.csv",
    index=False
)

In [22]:
print(df.duplicated().sum())

0


In [23]:
print(anomalies.shape[0])

75


In [24]:
#Cost Per Minute
df_clean['Cost_Per_Minute'] = (
    df_clean['Journey_Cost']
    /
    df_clean['Journey_Duration']
)

print(
    df_clean[
        [
            'Journey_Cost',
            'Journey_Duration',
            'Cost_Per_Minute'
        ]
    ].head()
)

   Journey_Cost  Journey_Duration  Cost_Per_Minute
0          3.97              21.0         0.189048
1          1.75              51.0         0.034314
2          9.40               2.0         4.700000
3          2.81              41.0         0.068537
4          2.98              24.0         0.124167


In [25]:
#Time Of Day
def classify_time(hour):

    if 6 <= hour < 10:
        return "Peak AM"

    elif 16 <= hour < 19:
        return "Peak PM"

    elif 10 <= hour < 16:
        return "Off-Peak"

    else:
        return "Night"

In [26]:
df_clean['Time_of_Day'] = (
    df_clean['Tap_In_Time']
    .dt.hour
    .apply(classify_time)
)

print(
    df_clean[
        [
            'Tap_In_Time',
            'Time_of_Day'
        ]
    ].head()
)

          Tap_In_Time Time_of_Day
0 2024-09-21 12:03:00    Off-Peak
1 2024-02-14 09:11:00     Peak AM
2 2024-08-25 08:03:00     Peak AM
3 2024-02-15 21:24:00       Night
4 2024-11-03 07:12:00     Peak AM


In [27]:
#Congestion AI Rule Engine
def congestion_tier(row):

    if (
        row['Time_of_Day'] == 'Peak AM'
        and
        row['Delay_Minutes'] > 15
        and
        row['Journey_Duration'] > 60
    ):
        return "Severe Friction"

    elif row['Delay_Minutes'] > 10:
        return "High Friction"

    elif row['Delay_Minutes'] > 5:
        return "Minor Delays"

    else:
        return "Smooth"

In [28]:
df_clean['Congestion_Tier'] = (
    df_clean.apply(
        congestion_tier,
        axis=1
    )
)

In [29]:
print(
    df_clean[
        [
            'Delay_Minutes',
            'Journey_Duration',
            'Time_of_Day',
            'Congestion_Tier'
        ]
    ].head()
)

   Delay_Minutes  Journey_Duration Time_of_Day Congestion_Tier
0              0              21.0    Off-Peak          Smooth
1             20              51.0     Peak AM   High Friction
2             14               2.0     Peak AM   High Friction
3              2              41.0       Night          Smooth
4              3              24.0     Peak AM          Smooth


In [30]:
#Distribution
print(
    df_clean['Congestion_Tier']
    .value_counts()
)

Congestion_Tier
Smooth             1026
High Friction       480
Minor Delays        354
Severe Friction      65
Name: count, dtype: int64


In [31]:
#Save Final Dataset
df_clean.to_csv(
    "feature_engineered_tfl_data.csv",
    index=False
)

print(
    "Feature engineering completed successfully."
)

Feature engineering completed successfully.


In [32]:
print(
    df_clean['Congestion_Tier']
    .value_counts()
)

Congestion_Tier
Smooth             1026
High Friction       480
Minor Delays        354
Severe Friction      65
Name: count, dtype: int64


In [33]:
import sqlite3
conn = sqlite3.connect(
    "tfl_commuter.db"
)
print("Database connected successfully.")

Database connected successfully.


In [34]:
df_final = pd.read_csv(
    "feature_engineered_tfl_data.csv"
)

print(df_final.head())

         Date    Journey_ID Transport_Mode Origin_Station Destination_Station  \
0  2024-09-21  TFL-J-422610           Tube      Stratford    Liverpool Street   
1  2024-02-14  TFL-J-228818            Bus     Paddington           Stratford   
2  2024-08-25  TFL-J-723520           Tube         Euston          Paddington   
3  2024-02-15  TFL-J-695985     Overground     Paddington            Victoria   
4  2024-11-03  TFL-J-175457           Tube         Euston           Stratford   

           Tap_In_Time         Tap_Out_Time  Journey_Cost  Delay_Minutes  \
0  2024-09-21 12:03:00  2024-09-21 12:24:00          3.97              0   
1  2024-02-14 09:11:00  2024-02-14 10:02:00          1.75             20   
2  2024-08-25 08:03:00  2024-08-25 08:05:00          9.40             14   
3  2024-02-15 21:24:00  2024-02-15 22:05:00          2.81              2   
4  2024-11-03 07:12:00  2024-11-03 07:36:00          2.98              3   

  Origin_Borough Destination_Borough  Journey_Duration  

In [35]:
#Push dataset into database
df_final.to_sql(
    "tfl_commuter_trends",
    conn,
    if_exists="replace",
    index=False
)
print("Table created successfully.")

Table created successfully.


In [ ]:
query = """
SELECT *
FROM tfl_commuter_trends
LIMIT 10;
"""
df_sql = pd.read_sql(
    query,
    conn
)
display(df_sql)

,Date,Journey_ID,Transport_Mode,Origin_Station,Destination_Station,Tap_In_Time,Tap_Out_Time,Journey_Cost,Delay_Minutes,Origin_Borough,Destination_Borough,Journey_Duration,Cost_Per_Minute,Time_of_Day,Congestion_Tier
0,2024-09-21,TFL-J-422610,Tube,Stratford,Liverpool Street,2024-09-21 12:03:00,2024-09-21 12:24:00,3.97,0,Newham,City of London,21.0,0.189048,Off-Peak,Smooth
1,2024-02-14,TFL-J-228818,Bus,Paddington,Stratford,2024-02-14 09:11:00,2024-02-14 10:02:00,1.75,20,Westminster,Newham,51.0,0.034314,Peak AM,High Friction
2,2024-08-25,TFL-J-723520,Tube,Euston,Paddington,2024-08-25 08:03:00,2024-08-25 08:05:00,9.40,14,Camden,Westminster,2.0,4.700000,Peak AM,High Friction
3,2024-02-15,TFL-J-695985,Overground,Paddington,Victoria,2024-02-15 21:24:00,2024-02-15 22:05:00,2.81,2,Westminster,Westminster,41.0,0.068537,Night,Smooth
4,2024-11-03,TFL-J-175457,Tube,Euston,Stratford,2024-11-03 07:12:00,2024-11-03 07:36:00,2.98,3,Camden,Newham,24.0,0.124167,Peak AM,Smooth
5,2024-08-25,TFL-J-961534,Tube,Euston,Kings Cross,2024-08-25 17:10:00,2024-08-25 17:57:00,2.86,3,Camden,Camden,47.0,0.060851,Peak PM,Smooth
6,2024-06-04,TFL-J-850930,Tube,Kings Cross,Euston,2024-06-04 13:56:00,2024-06-04 14:16:00,2.83,2,Camden,Camden,20.0,0.141500,Off-Peak,Smooth
7,2024-10-15,TFL-J-975541,Bus,Kings Cross,Bank,2024-10-15 07:12:00,2024-10-15 07:57:00,1.75,7,Camden,City of London,45.0,0.038889,Peak AM,Minor Delays
8,2024-07-11,TFL-J-752188,Tube,Euston,Bank,2024-07-11 07:36:00,2024-07-11 08:32:00,3.41,40,Camden,City of London,56.0,0.060893,Peak AM,High Friction
9,2024-08-12,TFL-J-537466,Bus,Victoria,Kings Cross,2024-08-12 18:09:00,2024-08-12 18:42:00,1.75,3,Westminster,Camden,33.0,0.053030,Peak PM,Smooth


In [41]:
#Query 1: Chronic Bottleneck Detector
query1 = """
SELECT
    Origin_Station,
    Transport_Mode,
    ROUND(
        AVG(Delay_Minutes),
        2
    ) AS Avg_Delay,
    ROUND(
        (
            SUM(
                CASE
                    WHEN Congestion_Tier =
                    'Severe Friction'
                    THEN 1
                    ELSE 0
                END
            ) * 100.0
        ) / COUNT(*),
        2
    ) AS Severe_Friction_Percentage
FROM tfl_commuter_trends
GROUP BY
    Origin_Station,
    Transport_Mode
HAVING
    AVG(Delay_Minutes) > 10
    AND
    (
        SUM(
            CASE
                WHEN Congestion_Tier =
                'Severe Friction'
                THEN 1
                ELSE 0
            END
        ) * 100.0
    ) / COUNT(*) > 20;
"""

In [38]:
df1 = pd.read_sql(
    query1,
    conn
)
display(df1)

,Origin_Station,Transport_Mode,Avg_Delay,Severe_Friction_Percentage


In [42]:
#•	Query 2: The "Overcharged Commuter" Vulnerability Finder
query2 = """
SELECT

    Journey_ID,
    Origin_Station,
    Destination_Station,
    Journey_Cost

FROM tfl_commuter_trends t1

WHERE Journey_Cost >

(
    SELECT
        AVG(t2.Journey_Cost) * 1.5

    FROM tfl_commuter_trends t2

    WHERE
        t1.Origin_Station =
        t2.Origin_Station

        AND

        t1.Destination_Station =
        t2.Destination_Station
);
"""

In [43]:
df2 = pd.read_sql(
    query2,
    conn
)
display(df2)

,Journey_ID,Origin_Station,Destination_Station,Journey_Cost
0,TFL-J-723520,Euston,Paddington,9.40
1,TFL-J-686066,Waterloo,Kings Cross,9.40
2,TFL-J-859248,Euston,Stratford,9.40
3,TFL-J-551978,Bank,Victoria,9.40
4,TFL-J-807126,London Bridge,Bank,5.14
...,...,...,...,...
162,TFL-J-268446,Kings Cross,Liverpool Street,9.25
163,TFL-J-711947,Waterloo,Euston,5.33
164,TFL-J-282943,London Bridge,Victoria,7.44
165,TFL-J-566378,Victoria,Paddington,9.40


In [44]:
#Query 3: Peak vs Off-Peak Stress Test
query3 = """
SELECT

    Origin_Station,
    Destination_Station,

    ROUND(
        AVG(
            CASE
                WHEN Time_of_Day = 'Peak AM'
                THEN Journey_Duration
            END
        ),
        2
    ) AS Peak_AM_Duration,

    ROUND(
        AVG(
            CASE
                WHEN Time_of_Day = 'Off-Peak'
                THEN Journey_Duration
            END
        ),
        2
    ) AS OffPeak_Duration

FROM tfl_commuter_trends

GROUP BY
    Origin_Station,
    Destination_Station

HAVING

    AVG(
        CASE
            WHEN Time_of_Day = 'Peak AM'
            THEN Journey_Duration
        END
    )

    >=

    2 *

    AVG(
        CASE
            WHEN Time_of_Day = 'Off-Peak'
            THEN Journey_Duration
        END
    );
"""

In [45]:
df3 = pd.read_sql(
    query3,
    conn
)
display(df3)

,Origin_Station,Destination_Station,Peak_AM_Duration,OffPeak_Duration
0,London Bridge,Paddington,45.33,21.5


In [46]:
#Query 4: Ghost Tap Audit
query4 = """
SELECT

    Origin_Station,

    COUNT(*) AS Total_Journeys,

    SUM(
        CASE
            WHEN Journey_Cost >= 9.40
            AND Journey_Duration < 15
            THEN 1
            ELSE 0
        END
    ) AS Ghost_Tap_Flags,

    ROUND(
        (
            SUM(
                CASE
                    WHEN Journey_Cost >= 9.40
                    AND Journey_Duration < 15
                    THEN 1
                    ELSE 0
                END
            ) * 100.0
        ) / COUNT(*),
        2
    ) AS Ghost_Tap_Percentage

FROM tfl_commuter_trends

GROUP BY Origin_Station

HAVING Ghost_Tap_Percentage > 0;
"""

In [47]:
df4 = pd.read_sql(
    query4,
    conn
)

display(df4)

,Origin_Station,Total_Journeys,Ghost_Tap_Flags,Ghost_Tap_Percentage
0,Bank,192,6,3.13
1,Canary Wharf,188,8,4.26
2,Euston,203,4,1.97
3,Kings Cross,180,4,2.22
4,Liverpool Street,195,7,3.59
5,London Bridge,177,5,2.82
6,Paddington,195,1,0.51
7,Stratford,192,6,3.13
8,Victoria,212,5,2.36
9,Waterloo,191,4,2.09


In [48]:
df1.to_csv("query1_bottlenecks.csv", index=False)
df2.to_csv("query2_overcharged.csv", index=False)
df3.to_csv("query3_peak_stress.csv", index=False)
df4.to_csv("query4_ghost_tap.csv", index=False)